## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Resolve project root by walking up until 'src/' is found
_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import torch
import yaml
from ultralytics import YOLO

from src.ppe_detection.utils import (
    MODELS_DIR, EXPERIMENTS_DIR, ensure_dirs, write_dataset_yaml
)
from src.ppe_detection.trainer import (
    train_ppe_model, detect_train_device, recommended_train_config
)
ensure_dirs()

## 2. Environment Check

> **Apple Silicon note:** training runs on **CPU**, not MPS. YOLOv8's backward
> pass crashes on the MPS backend with torch ≤ 2.5
> (`view size is not compatible with input tensor's size and stride`), a known
> PyTorch regression. `detect_train_device()` skips MPS for training; inference
> (notebooks 05/06) still uses MPS via `detect_device()`. On a CUDA machine the
> GPU is used automatically.

In [ ]:
import platform

print("=" * 50)
print("  TRAINING ENVIRONMENT")
print("=" * 50)
print(f"  PyTorch      : {torch.__version__}")

cuda_ok = torch.cuda.is_available()
print(f"  CUDA         : {cuda_ok}")
if cuda_ok:
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM         : {vram:.1f} GB")
elif platform.system() == "Darwin":
    print(f"  MPS (Apple)  : {torch.backends.mps.is_available()}  (skipped for training)")

# detect_train_device() returns CUDA when present, else CPU (never MPS) — see note above.
DEVICE = detect_train_device()
print(f"  Train device : {DEVICE}")
print("=" * 50)

## 3. Dataset Configuration

`write_dataset_yaml()` regenerates the dataset config with an **absolute** `path`
resolved from `src/ppe_detection/utils.py`. This avoids Ultralytics' relative-path
trap (it resolves `path:` against its global `datasets_dir`, not the YAML location),
which otherwise makes training fail with *"images not found"*.

In [ ]:
DATA_YAML = write_dataset_yaml()   # absolute-path config, resolved from utils
print(f"Config: {DATA_YAML}")
print(f"Exists: {DATA_YAML.exists()}")
print()

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print("DATASET YAML CONTENTS")
print("=" * 50)
print(f"  path  : {cfg['path']}")
print(f"  train : {cfg['train']}")
print(f"  val   : {cfg['val']}")
print(f"  test  : {cfg['test']}")
print(f"  nc    : {cfg['nc']} classes")
print()
print("  Classes:")
_names = cfg["names"]
_items = _names.items() if isinstance(_names, dict) else enumerate(_names)
for k, v in _items:
    print(f"    {k:2d}: {v}")

## 4. Training Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Base model | `yolov8n.pt` | COCO pre-trained, fastest iteration |
| Epochs | 10 (CPU) / 100 (GPU) | CPU bounded; GPU aims for mAP>0.70 (early stop via patience) |
| Image size | 416 (CPU) / 640 (GPU) | Lower res keeps CPU epochs to minutes |
| Data fraction | 0.20 (CPU) / 1.0 (GPU) | Subset keeps CPU training time bounded |
| Batch | 16 (CPU) / −1 AutoBatch (GPU) | AutoBatch is CUDA-only; picks the largest batch for VRAM |
| Optimizer | AdamW | Ultralytics default — best convergence on small datasets |
| Augmentation | Built-in | Mosaic, mixup, copy-paste, flips, HSV jitter |
| Patience | 50 | Early stopping if no improvement for 50 epochs |

These values come from `recommended_train_config(DEVICE)` (in
`src/ppe_detection/trainer.py`) — **the notebook auto-selects the CPU or CUDA
profile**, so a collaborator on a GPU desktop and one on a CPU laptop both run
this unchanged. Override `cfg` in the next cell to customise.

> **GPU setup:** install a CUDA-enabled PyTorch (e.g. `conda env create -f
> environment.yml` on a CUDA box, or follow pytorch.org for the right CUDA
> build). When `torch.cuda.is_available()` is `True`, training uses the GPU
> automatically — no notebook edits needed.

> **Reproducibility:** All training args are saved automatically to
> `experiments/smartmine_v1/baseline/args.yaml` by Ultralytics.

## 5. Launch Training

In [ ]:
# Device-aware config so the same notebook runs on CPU and GPU without edits:
#   • CUDA  → full dataset @640, AutoBatch (-1), 100 epochs (aims for mAP>0.70)
#   • CPU/MPS → 20% subset @416, batch 16, 10 epochs (~30 min, not hours)
# Override any field before training, e.g.:
#   cfg.update(fraction=1.0, imgsz=640, epochs=50)   # CPU user wanting full run
#   cfg.update(epochs=50)                            # GPU user in a hurry
cfg = recommended_train_config(DEVICE)
print(f"Device: {DEVICE}   training config: {cfg}")

best_weights = train_ppe_model(
    data_yaml  = DATA_YAML,
    base_model = "yolov8n.pt",
    name       = "baseline",
    device     = DEVICE,
    **cfg,
)
print(f"\n✅ Training complete.")
print(f"   Best weights → {best_weights}")

## 6. Training Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

EXP_DIR = EXPERIMENTS_DIR / "smartmine_v1" / "baseline"

results_png = EXP_DIR / "results.png"
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(18, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Curves — Loss & mAP", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(EXPERIMENTS_DIR / "smartmine_v1" / "training_curves.png"), dpi=150)
    plt.show()
else:
    print(f"Run training first. Expected: {results_png}")

In [ ]:
# Show confusion matrix from training validation
cm_png = EXP_DIR / "confusion_matrix.png"
if cm_png.exists():
    img = mpimg.imread(str(cm_png))
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix — Validation Set", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrix not yet available.")

In [ ]:
# Load and print best metrics from results.csv
import pandas as pd
results_csv = EXP_DIR / "results.csv"
if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    results_df.columns = results_df.columns.str.strip()
    last = results_df.iloc[-1]
    best_epoch = results_df["metrics/mAP50(B)"].idxmax()
    best = results_df.iloc[best_epoch]

    print("FINAL EPOCH METRICS")
    print("=" * 50)
    for col in results_df.columns:
        if "metrics" in col or "loss" in col.lower():
            print(f"  {col:<35}: {last[col]:.4f}")
    print()
    print(f"BEST EPOCH: {int(best_epoch) + 1}")
    print(f"  mAP50    : {best['metrics/mAP50(B)']:.4f}")
    print(f"  mAP50-95 : {best['metrics/mAP50-95(B)']:.4f}")
else:
    print("Results CSV not yet available — run training first.")

## 7. Conclusions & Next Steps

**Training checklist:**
- [ ] Loss curves converging (box_loss, cls_loss, dfl_loss decreasing)
- [ ] mAP50 > 0.70 on validation
- [ ] No obvious overfitting (val loss not rising while train loss falls)
- [ ] Best weights saved to `models/ppe/`

**If mAP50 < 0.70:**
- Try `yolov8s.pt` (more capacity)
- Increase epochs to 150
- Check class imbalance in `01_dataset_exploration.ipynb`

**Next:** `04_evaluation.ipynb` — comprehensive metrics on the held-out test set.